In [10]:
import sys
import time
from importlib import reload
from pathlib import Path

sys.path.append("../code/src")

import inference
import torch
import xarray as xr

reload(inference)

dataset = xr.open_zarr("../data/processed/antarctic/all/all.zarr")
dataset = dataset.sel(
    {
        "feature": [
            "siconc",
            "sithic",
            "sivelv",
            "sivelu",
            "utau_ai",
            "utau_oi",
            "vtau_ai",
            "vtau_oi",
        ]
    }
)

data_subset = dataset.isel(z=slice(None, None, 1000))

t = time.time()
model = inference.RheologyModel(
    Path("../results/nn/arctic-tests/all/all/20260214_2119"),
    Path("../code"),
    data_subset,
    "cuda",
)
print(f"{time.time() - t:.2f}s to import model")

torch_dataset = torch.utils.data.TensorDataset(torch.tensor(dataset.features.values.T))
dataloader = torch.utils.data.DataLoader(torch_dataset, 10000)

t = time.time()
outputs = []
for (inputs,) in dataloader:
    outputs.append(model(inputs))
outputs = torch.cat(outputs)
print(f"{time.time() - t:.2f}s to perform inference on {dataset.sizes['z']} points")

<xarray.Dataset> Size: 8MB
Dimensions:        (d_label: 2, z: 81676, feature: 8, label: 2)
Coordinates:
  * d_label        (d_label) <U8 64B 'd_sivelv' 'd_sivelu'
    lat            (z) float32 327kB dask.array<chunksize=(495,), meta=np.ndarray>
    lon            (z) float32 327kB dask.array<chunksize=(495,), meta=np.ndarray>
    pair           (z) int64 653kB dask.array<chunksize=(248,), meta=np.ndarray>
    time_features  (z) object 653kB dask.array<chunksize=(248,), meta=np.ndarray>
    time_labels    (z) object 653kB dask.array<chunksize=(248,), meta=np.ndarray>
    x              (z) int64 653kB dask.array<chunksize=(248,), meta=np.ndarray>
    y              (z) int64 653kB dask.array<chunksize=(248,), meta=np.ndarray>
  * feature        (feature) <U7 224B 'siconc' 'sithic' ... 'vtau_ai' 'vtau_oi'
  * label          (label) <U6 48B 'sivelv' 'sivelu'
Dimensions without coordinates: z
Data variables:
    d_labels       (d_label, z) float32 653kB dask.array<chunksize=(1, 1737), met